# 11 - RAPIDS cuDF — **Soluciones**

Soluciones a los ejercicios del cuaderno 11 (con manejo de ausencia de cuDF).

## Contexto (recrear dataset)

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 100_000
pdf = pd.DataFrame({
    "id": np.arange(n),
    "x": rng.normal(0,1,size=n),
    "y": rng.integers(0, 10, size=n),
    "city": rng.choice(["Cádiz","Sevilla","Málaga","Córdoba"], size=n)
})

try:
    import cudf
    has_cudf = True
    gdf = cudf.from_pandas(pdf)
except Exception as e:
    has_cudf = False
    print("cuDF NO disponible:", e)

## Ejercicio 1
**Columna `w = x*y` y media por `city` (cuDF)**

In [ ]:
if has_cudf:
    gdf = gdf.assign(w = gdf["x"] * gdf["y"])
    media_w_city_g = gdf.groupby("city")["w"].mean().reset_index().sort_values("w", ascending=False)
    media_w_city_g
else:
    print("Sin cuDF: muestra solución equivalente en pandas en el siguiente bloque.")

## Ejercicio 2
**Replica en pandas y compara resultados**

In [ ]:
pdf2 = pdf.copy()
pdf2["w"] = pdf2["x"] * pdf2["y"]
media_w_city_p = pdf2.groupby("city")["w"].mean().reset_index().sort_values("w", ascending=False)
media_w_city_p

**Comparación rápida (si cuDF disponible):**

In [ ]:
if 'media_w_city_g' in globals():
    # Convertir cuDF -> pandas para comparar
    try:
        media_w_city_g_pd = media_w_city_g.to_pandas()
    except Exception:
        media_w_city_g_pd = media_w_city_g  # ya es pandas si no hay cuDF
    print("¿Coinciden las ciudades? ", set(media_w_city_g_pd["city"]) == set(media_w_city_p["city"]))
    # Comparación por unión y diferencia de medias (tolerancia)
    merged = media_w_city_p.merge(media_w_city_g_pd, on="city", suffixes=("_pd","_gd"))
    merged["diff"] = (merged["w_pd"] - merged["w_gd"]).abs()
    merged
else:
    print("Solo se muestra el resultado en pandas.")

## Ejercicio 3 (opcional)
**Escala a 1 millón de filas y repite la comparativa**

In [ ]:
# Este bloque es opcional y depende de los recursos del entorno.
import numpy as np, pandas as pd
m = 1_000_000
pdf_big = pd.DataFrame({
    "id": np.arange(m),
    "x": np.random.normal(size=m),
    "y": np.random.randint(0, 10, size=m),
    "city": np.random.choice(["Cádiz","Sevilla","Málaga","Córdoba"], size=m)
})
# pandas
res_pd = (pdf_big.assign(w=lambda d: d["x"]*d["y"])
                  .groupby("city")["w"]
                  .mean()
                  .sort_values(ascending=False))
res_pd.head()

---

## ✅ Autotests 11

In [ ]:
def check(cond, msg_ok, msg_fail):
    if cond:
        print("✔", msg_ok)
    else:
        raise AssertionError("✘ " + msg_fail)

# Test 1: solución en pandas existe y tiene columna 'w'
check(("media_w_city_p" in globals()) and ("w" in media_w_city_p.columns),
      "media_w_city_p generado con columna 'w'.",
      "Falta 'media_w_city_p' o columna 'w'.")

# Test 2: si hay cuDF, existe media_w_city_g y ciudades compatibles
if 'has_cudf' in globals() and has_cudf and ('media_w_city_g' in globals()):
    try:
        media_w_city_g_pd = media_w_city_g.to_pandas()
    except Exception:
        media_w_city_g_pd = media_w_city_g
    check(set(media_w_city_g_pd["city"]) == set(media_w_city_p["city"]),
          "Con cuDF: mismas ciudades presentes en ambos resultados.",
          "CuDF vs pandas: discrepancia de ciudades.")
else:
    print("ℹ cuDF no disponible: se validan solo resultados en pandas.")

# Test 3: tamaños razonables
check(len(media_w_city_p) > 0, "Resultado pandas no vacío.", "Resultado pandas vacío inesperadamente.")